# Day 30 — Day 26 Dashboard Repair, URL Integration and Requested Features

This is the **single notebook** for the remaining integration work. You do not need to run a separate Day 29 notebook.

It repairs URL registration and updates the **Day 26 dashboard view** (`dashboard_views_day26.py`) and its actual template. It is designed to restore the Day 26 dashboard at `/api/dashboard/` and keep the Day 28 prediction centre at `/api/predictions/`.

The dashboard includes:
1. Dashboard Overview
2. Player Search
3. Batting Predictions — separate ML and DL panels
4. Bowling Predictions — separate target controls
5. Model Comparison — saved evaluation metrics
6. EDA & Analytics
7. Batch Analysis — JSON request and downloadable result
8. Data Ingestion & Status

It creates backups before changing files and generates documentation for every generated or modified file. The notebook stops if the expected Day 26 view/template cannot be verified.

**Important:** DL inference and ingestion actions are not claimed to work unless their backend endpoints actually exist. Run this notebook from the repository root using the `ipl-player-performance` Conda environment.


In [1]:
from pathlib import Path
import datetime
import shutil
import subprocess
import sys
import re
import csv
import hashlib

CWD = Path.cwd().resolve()
ROOT = next(
    (p for p in [CWD, *CWD.parents]
     if (p / "django_api" / "manage.py").exists()
     and (p / "django_api" / "ipl_api" / "urls.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Repository root not found. Set notebook working directory to "
        r"D:\Git\Personal\ipl-player-performance"
    )

DJANGO_DIR = ROOT / "django_api"
APP_DIR = DJANGO_DIR / "ipl_api"
CONFIG_URLS = DJANGO_DIR / "config" / "urls.py"
URLS_PATH = APP_DIR / "urls.py"
DASHBOARD_VIEW_PATH = APP_DIR / "dashboard_views_day26.py"
PREDICTION_VIEW_PATH = APP_DIR / "prediction_center_views_day28.py"
TEMPLATE_PATH = APP_DIR / "templates" / "ipl_api" / "dashboard_day25.html"
JS_PATH = APP_DIR / "static" / "ipl_api" / "js" / "dashboard_day25.js"
CSS_PATH = APP_DIR / "static" / "ipl_api" / "css" / "dashboard_day25.css"
BACKUP_DIR = ROOT / "backups" / ("day30_dashboard_repair_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
DOC_DIR = ROOT / "docs" / "day30"

print("Repository:", ROOT)
print("Django directory:", DJANGO_DIR)
print("App URL file:", URLS_PATH)
print("Day 26 dashboard view:", DASHBOARD_VIEW_PATH)
print("Target template:", TEMPLATE_PATH)
print("Backup directory:", BACKUP_DIR)


Repository: D:\Git\Personal\ipl-player-performance
Django directory: D:\Git\Personal\ipl-player-performance\django_api
App URL file: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
Day 26 dashboard view: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\dashboard_views_day26.py
Target template: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html
Backup directory: D:\Git\Personal\ipl-player-performance\backups\day30_dashboard_repair_20261023_215041


## 1. Inspect current routes and back up files

In [2]:
# Preflight the Day 26 view and infer its template path before editing anything.
for required in [CONFIG_URLS, URLS_PATH, DASHBOARD_VIEW_PATH]:
    if not required.exists():
        raise FileNotFoundError(f"Required project file missing: {required}")

view_text = DASHBOARD_VIEW_PATH.read_text(encoding="utf-8")
print("Day 26 view source:\n")
print(view_text)

# Parse render(request, "app/template.html") or render(request, 'app/template.html').
render_match = re.search(r"""render\s*\(\s*request\s*,\s*["']([^"']+)["']""", view_text)
if not render_match:
    raise RuntimeError(
        "Could not determine the template rendered by dashboard_views_day26.py. "
        "No project files have been changed. Inspect this view before proceeding."
    )
template_name = render_match.group(1)
if not template_name.startswith("ipl_api/"):
    raise RuntimeError(f"Unexpected Day 26 template name: {template_name}")
TEMPLATE_PATH = APP_DIR / "templates" / template_name
print("\nConfirmed Day 26 template:", TEMPLATE_PATH)

if not TEMPLATE_PATH.exists():
    raise FileNotFoundError(
        f"The Day 26 view renders {template_name}, but that template does not exist. "
        "Restore it before running the dashboard update."
    )

current_app_urls = URLS_PATH.read_text(encoding="utf-8")
current_config_urls = CONFIG_URLS.read_text(encoding="utf-8")
print("\nCurrent app URLs:\n", current_app_urls)
print("\nCurrent project URLs:\n", current_config_urls)

# Back up all files that this notebook may modify.
for file_path in [CONFIG_URLS, URLS_PATH, TEMPLATE_PATH, JS_PATH, CSS_PATH]:
    if file_path.exists():
        destination = BACKUP_DIR / file_path.relative_to(ROOT)
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, destination)
        print("Backup:", destination)


Day 26 view source:

from django.shortcuts import render

def dashboard_day25(request):
    """Render the IPL analytics dashboard shell."""
    return render(request, "ipl_api/dashboard_day25.html")


Confirmed Day 26 template: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html

Current app URLs:
 from django.urls import path

from .views import PlayerListAPIView, PlayerDetailAPIView
from .eda_summary_day26 import eda_summary_day26
from .eda_views import (
    EDABattingAPIView,
    EDABowlingAPIView,
    EDASeasonAPIView,
    EDAVenueAPIView,
    EDAOppositionAPIView,
    EDARecentFormAPIView,
)
from .prediction_views import (
    BattingPredictionAPIView,
    BowlingPredictionAPIView,
    UnifiedPredictionAPIView,
)
from .batch_prediction_views import BatchPredictionAPIView
from .dashboard_views_day28 import dashboard_day28
from .prediction_center_views_day28 import prediction_center_day28
from .player_name_prediction_views_day28 import (

## 2. Repair URL registration and write the eight-section UI into the Day 26 dashboard template

In [3]:
# URL repair is included here (the separate Day 29 notebook is not required).
config_text = """from django.contrib import admin
from django.urls import include, path
from django.views.generic.base import RedirectView

urlpatterns = [
    path("", RedirectView.as_view(url="/api/dashboard/", permanent=False), name="home"),
    path("api/", include("ipl_api.urls")),
    path("admin/", admin.site.urls),
]
"""

# Preserve the project's known API modules and remove the duplicate predictions/ route.
# The Day 26 dashboard is /api/dashboard/; the Day 28 prediction centre is /api/predictions/.
app_text = """from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView
from .dashboard_views_day26 import dashboard_day25
from .eda_summary_day26 import eda_summary_day26
from .eda_views import EDABattingAPIView, EDABowlingAPIView, EDASeasonAPIView, EDAVenueAPIView, EDAOppositionAPIView, EDARecentFormAPIView
from .prediction_views import BattingPredictionAPIView, BowlingPredictionAPIView, UnifiedPredictionAPIView
from .batch_prediction_views import BatchPredictionAPIView
from .prediction_center_views_day28 import prediction_center_day28
from .player_name_prediction_views_day28 import player_name_prediction_day28, player_name_options_day28, model_metrics_day28

urlpatterns = [
    path("dashboard/", dashboard_day25, name="ipl-dashboard"),
    path("predictions/", prediction_center_day28, name="ipl-predictions"),
    path("players/", PlayerListAPIView.as_view(), name="player-list"),
    path("players/search-options/", player_name_options_day28, name="player-name-options-day28"),
    path("players/<str:player_name>/", PlayerDetailAPIView.as_view(), name="player-detail"),
    path("eda/summary/", eda_summary_day26, name="eda-summary"),
    path("eda/batting/", EDABattingAPIView.as_view(), name="eda-batting"),
    path("eda/bowling/", EDABowlingAPIView.as_view(), name="eda-bowling"),
    path("eda/season/", EDASeasonAPIView.as_view(), name="eda-season"),
    path("eda/venue/", EDAVenueAPIView.as_view(), name="eda-venue"),
    path("eda/opposition/", EDAOppositionAPIView.as_view(), name="eda-opposition"),
    path("eda/recent-form/", EDARecentFormAPIView.as_view(), name="eda-recent-form"),
    path("eda/model-metrics/", model_metrics_day28, name="model-metrics"),
    path("predict/", UnifiedPredictionAPIView.as_view(), name="predict-unified"),
    path("predict/batting/", BattingPredictionAPIView.as_view(), name="predict-batting"),
    path("predict/bowling/", BowlingPredictionAPIView.as_view(), name="predict-bowling"),
    path("predict/batch/", BatchPredictionAPIView.as_view(), name="predict-batch"),
    path("predict/player/", player_name_prediction_day28, name="predict-player-name-day28"),
]
"""

# Confirm every imported module exists before writing the URL files.
for module_file in [
    APP_DIR / "views.py", APP_DIR / "dashboard_views_day26.py",
    APP_DIR / "eda_summary_day26.py", APP_DIR / "eda_views.py",
    APP_DIR / "prediction_views.py", APP_DIR / "batch_prediction_views.py",
    APP_DIR / "prediction_center_views_day28.py",
    APP_DIR / "player_name_prediction_views_day28.py",
]:
    if not module_file.exists():
        raise FileNotFoundError(f"Required view module missing; URL files not changed: {module_file}")

CONFIG_URLS.write_text(config_text, encoding="utf-8")
URLS_PATH.write_text(app_text, encoding="utf-8")
print("Repaired project URLs:", CONFIG_URLS)
print("Repaired app URLs:", URLS_PATH)

TEMPLATE_PATH.parent.mkdir(parents=True, exist_ok=True)
JS_PATH.parent.mkdir(parents=True, exist_ok=True)
CSS_PATH.parent.mkdir(parents=True, exist_ok=True)

template = r'''{% load static %}
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>IPL Player Analytics</title>
  <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
  <link href="{% static 'ipl_api/css/dashboard_day25.css' %}" rel="stylesheet">
</head>
<body>
<nav class="navbar navbar-dark bg-dark px-3">
  <a class="navbar-brand" href="/api/dashboard/">IPL Player Analytics</a>
  <span class="text-light small">Performance • Analytics • Prediction</span>
</nav>
<main class="container-fluid py-4">
  <div class="d-flex flex-wrap justify-content-between align-items-center mb-3">
    <div><h1 class="h3 mb-1">IPL Analytics Workspace</h1><p class="text-secondary mb-0">Historical performance, model estimates and batch analysis.</p></div>
    <button class="btn btn-outline-secondary" id="refreshOverview">Refresh overview</button>
  </div>
  <div id="globalMessage" class="alert alert-secondary" role="status">Loading dashboard overview…</div>

  <ul class="nav nav-pills flex-wrap gap-2 mb-4" id="sectionTabs">
    <li class="nav-item"><button class="nav-link active" data-target="overview">1. Overview</button></li>
    <li class="nav-item"><button class="nav-link" data-target="player-search">2. Player Search</button></li>
    <li class="nav-item"><button class="nav-link" data-target="batting">3. Batting Predictions</button></li>
    <li class="nav-item"><button class="nav-link" data-target="bowling">4. Bowling Predictions</button></li>
    <li class="nav-item"><button class="nav-link" data-target="models">5. Model Comparison</button></li>
    <li class="nav-item"><button class="nav-link" data-target="eda">6. EDA & Analytics</button></li>
    <li class="nav-item"><button class="nav-link" data-target="batch">7. Batch Analysis</button></li>
    <li class="nav-item"><button class="nav-link" data-target="ingestion">8. Ingestion & Status</button></li>
  </ul>

  <section class="feature-section active" id="overview">
    <h2 class="h4">Dashboard Overview</h2>
    <div class="row g-3" id="overviewCards">
      <div class="col-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Players</div><div class="metric-value" id="playersMetric">—</div></div></div></div>
      <div class="col-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Matches</div><div class="metric-value" id="matchesMetric">—</div></div></div></div>
      <div class="col-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Batting records</div><div class="metric-value" id="battingMetric">—</div></div></div></div>
      <div class="col-6 col-xl-3"><div class="card metric-card"><div class="card-body"><div class="text-secondary">Bowling records</div><div class="metric-value" id="bowlingMetric">—</div></div></div></div>
    </div>
    <p class="text-secondary mt-3">Counts are populated from the registered EDA summary endpoint.</p>
  </section>

  <section class="feature-section" id="player-search">
    <h2 class="h4">Player Search & Historical Performance</h2>
    <div class="row g-3"><div class="col-lg-5">
      <label class="form-label" for="playerName">Player name</label>
      <input class="form-control" id="playerName" list="playerOptions" placeholder="Start typing a player name">
      <datalist id="playerOptions"></datalist>
      <button class="btn btn-primary mt-2" id="searchPlayer">Search player</button>
    </div><div class="col-lg-7"><pre class="result-box" id="playerResult">Search for a player to see historical summary.</pre></div></div>
  </section>

  <section class="feature-section" id="batting">
    <h2 class="h4">Batting Predictions</h2>
    <div class="row g-3">
      <div class="col-xl-6"><div class="card h-100"><div class="card-header">ML — Single prediction</div><div class="card-body">
        <label class="form-label" for="battingPlayer">Player name (if supported by your registered player-name API)</label>
        <input class="form-control mb-2" id="battingPlayer" placeholder="Player name">
        <label class="form-label" for="battingJson">Or provide the exact feature JSON expected by the model</label>
        <textarea class="form-control code-input" id="battingJson" placeholder='{"feature_name": 0}'></textarea>
        <button class="btn btn-primary mt-2" id="runBattingML">Run ML prediction</button>
        <pre class="result-box mt-3" id="battingMLResult">No batting prediction requested.</pre>
      </div></div></div>
      <div class="col-xl-6"><div class="card h-100"><div class="card-header">DL — Single prediction</div><div class="card-body">
        <div class="alert alert-warning" id="battingDLStatus">DL inference is not enabled until a dedicated sequence-model endpoint is registered and validated.</div>
        <label class="form-label" for="battingDLJson">Five-match sequence JSON (for a future DL endpoint)</label>
        <textarea class="form-control code-input" id="battingDLJson" placeholder='Provide the exact documented sequence payload only after DL endpoint is implemented.'></textarea>
        <button class="btn btn-secondary mt-2" disabled>DL endpoint not configured</button>
        <pre class="result-box mt-3" id="battingDLResult">Unavailable — no verified DL endpoint.</pre>
      </div></div></div>
    </div>
  </section>

  <section class="feature-section" id="bowling">
    <h2 class="h4">Bowling Predictions</h2>
    <div class="row g-3">
      <div class="col-xl-6"><div class="card h-100"><div class="card-header">ML — Single prediction</div><div class="card-body">
        <label class="form-label" for="bowlingPlayer">Player name (if supported by your registered player-name API)</label>
        <input class="form-control mb-2" id="bowlingPlayer" placeholder="Player name">
        <label class="form-label" for="bowlingJson">Or exact feature JSON</label>
        <textarea class="form-control code-input" id="bowlingJson" placeholder='{"feature_name": 0}'></textarea>
        <button class="btn btn-primary mt-2" id="runBowlingML">Run ML prediction</button>
        <pre class="result-box mt-3" id="bowlingMLResult">No bowling prediction requested.</pre>
      </div></div></div>
      <div class="col-xl-6"><div class="card h-100"><div class="card-header">DL — Single prediction</div><div class="card-body">
        <div class="alert alert-warning">DL inference is not enabled until a dedicated sequence-model endpoint is registered and validated.</div>
        <label class="form-label" for="bowlingDLJson">Five-match sequence JSON (for a future DL endpoint)</label>
        <textarea class="form-control code-input" id="bowlingDLJson" placeholder="Exact documented sequence payload required."></textarea>
        <button class="btn btn-secondary mt-2" disabled>DL endpoint not configured</button>
        <pre class="result-box mt-3">Unavailable — no verified DL endpoint.</pre>
      </div></div></div>
    </div>
  </section>

  <section class="feature-section" id="models">
    <h2 class="h4">Model Comparison</h2>
    <p>Displays saved evaluation metrics only when the metrics endpoint returns them. It does not invent or recalculate model scores in the browser.</p>
    <button class="btn btn-primary" id="loadMetrics">Load saved model metrics</button>
    <pre class="result-box mt-3" id="metricsResult">Metrics not loaded.</pre>
  </section>

  <section class="feature-section" id="eda">
    <h2 class="h4">EDA & Analytics</h2>
    <div class="row g-2 align-items-end">
      <div class="col-md-5"><label class="form-label" for="edaEndpoint">Analysis</label>
        <select class="form-select" id="edaEndpoint">
          <option value="/api/eda/summary/">Summary</option><option value="/api/eda/batting/">Batting</option>
          <option value="/api/eda/bowling/">Bowling</option><option value="/api/eda/season/">Season</option>
          <option value="/api/eda/venue/">Venue</option><option value="/api/eda/opposition/">Opposition</option>
          <option value="/api/eda/recent-form/">Recent form</option>
        </select>
      </div>
      <div class="col-md-3"><button class="btn btn-primary" id="loadEDA">Load analysis</button></div>
    </div>
    <pre class="result-box mt-3" id="edaResult">Select an analysis and load it.</pre>
  </section>

  <section class="feature-section" id="batch">
    <h2 class="h4">Batch Player Analysis</h2>
    <p>Submit the documented batch endpoint payload. The feature JSON must follow the selected target's model manifest; player names alone work only if the backend explicitly supports them.</p>
    <div class="row g-3">
      <div class="col-lg-6"><label class="form-label" for="batchEndpoint">Batch endpoint</label>
        <select class="form-select mb-2" id="batchEndpoint"><option value="/api/predict/batch/">ML batch prediction</option></select>
        <label class="form-label" for="batchJson">Batch JSON payload</label>
        <textarea class="form-control code-input" id="batchJson" placeholder='Paste the exact payload expected by the batch API'></textarea>
        <button class="btn btn-primary mt-2" id="runBatch">Run batch analysis</button> <button class="btn btn-outline-secondary mt-2" id="downloadBatch" hidden>Download JSON results</button>
      </div><div class="col-lg-6"><pre class="result-box" id="batchResult">No batch request submitted.</pre></div>
    </div>
  </section>

  <section class="feature-section" id="ingestion">
    <h2 class="h4">Data Ingestion & Status</h2>
    <p>This section reports the availability of registered endpoints. It will not trigger a data import unless a verified ingestion endpoint exists.</p>
    <button class="btn btn-outline-primary" id="checkStatus">Check ingestion/status endpoints</button>
    <pre class="result-box mt-3" id="statusResult">Status not checked.</pre>
  </section>
</main>
<script src="{% static 'ipl_api/js/dashboard_day25.js' %}"></script>
</body>
</html>
'''
TEMPLATE_PATH.write_text(template, encoding="utf-8")
print("Wrote:", TEMPLATE_PATH.relative_to(ROOT))


Repaired project URLs: D:\Git\Personal\ipl-player-performance\django_api\config\urls.py
Repaired app URLs: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\urls.py
Wrote: django_api\ipl_api\templates\ipl_api\dashboard_day25.html


## 3. Add JavaScript for navigation, API calls, and honest status handling

In [4]:
js = r'''(() => {
  "use strict";
  const $ = id => document.getElementById(id);
  const jsonText = value => JSON.stringify(value, null, 2);
  const show = (id, value) => { const el = $(id); if (el) el.textContent = typeof value === "string" ? value : jsonText(value); };
  const safeParse = (raw, label) => {
    try { return JSON.parse(raw); }
    catch (err) { throw new Error(label + " is not valid JSON: " + err.message); }
  };
  async function request(url, options = {}) {
    const response = await fetch(url, {
      credentials: "same-origin",
      headers: { "Accept": "application/json", ...(options.body ? {"Content-Type":"application/json"} : {}), ...(options.headers || {}) },
      ...options
    });
    const raw = await response.text();
    let body;
    try { body = raw ? JSON.parse(raw) : null; } catch { body = raw; }
    if (!response.ok) {
      const detail = body && typeof body === "object" ? (body.detail || body.error || body.message || body) : body;
      throw new Error("HTTP " + response.status + " from " + url + ": " + (typeof detail === "string" ? detail : jsonText(detail)));
    }
    return body;
  }
  function setMessage(message, kind = "secondary") {
    const el = $("globalMessage");
    if (el) { el.className = "alert alert-" + kind; el.textContent = message; }
  }
  document.querySelectorAll("#sectionTabs [data-target]").forEach(button => {
    button.addEventListener("click", () => {
      document.querySelectorAll("#sectionTabs .nav-link").forEach(b => b.classList.remove("active"));
      document.querySelectorAll(".feature-section").forEach(s => s.classList.remove("active"));
      button.classList.add("active");
      $(button.dataset.target).classList.add("active");
    });
  });

  async function loadOverview() {
    try {
      const data = await request("/api/eda/summary/");
      const obj = data && data.results ? data.results : data;
      const aliases = {
        playersMetric: ["players", "player_count", "total_players"],
        matchesMetric: ["matches", "match_count", "total_matches"],
        battingMetric: ["batting_records", "batting_count"],
        bowlingMetric: ["bowling_records", "bowling_count"]
      };
      for (const [id, keys] of Object.entries(aliases)) {
        const key = keys.find(k => obj && obj[k] !== undefined);
        $(id).textContent = key ? obj[key] : "—";
      }
      setMessage("Summary API reachable. Counts reflect the response fields available from the backend.", "success");
    } catch (err) { setMessage("Summary API error: " + err.message, "danger"); }
  }
  $("refreshOverview").addEventListener("click", loadOverview);
  loadOverview();

  $("searchPlayer").addEventListener("click", async () => {
    const name = $("playerName").value.trim();
    if (!name) { show("playerResult", "Enter a player name."); return; }
    try { show("playerResult", await request("/api/players/" + encodeURIComponent(name) + "/")); }
    catch (err) { show("playerResult", err.message); }
  });

  // Populate player suggestions only if the configured endpoint supports this GET contract.
  (async () => {
    try {
      const data = await request("/api/players/search-options/");
      const rows = Array.isArray(data) ? data : (data.results || data.players || []);
      for (const row of rows) {
        const value = typeof row === "string" ? row : (row.player_name || row.name);
        if (value) { const option = document.createElement("option"); option.value = value; $("playerOptions").appendChild(option); }
      }
    } catch (err) { console.info("Player suggestions not available:", err.message); }
  })();

  async function prediction(endpoint, playerId, jsonId, resultId) {
    const player = $(playerId).value.trim();
    const raw = $(jsonId).value.trim();
    try {
      let payload;
      if (raw) payload = safeParse(raw, "Feature payload");
      else if (player) {
        const target = endpoint.includes("batting") ? "next_match_runs" :
          (endpoint.includes("bowling") ? "next_match_wickets" : "");
        if (!target) throw new Error("Select a supported target.");
        payload = {player_name: player, target: target};
        show(resultId, await request("/api/predict/player/", {method:"POST", body:JSON.stringify(payload)}));
        return;
      } else throw new Error("Enter a player name or provide feature JSON.");
      show(resultId, await request(endpoint, {method:"POST", body:JSON.stringify(payload)}));
    } catch (err) { show(resultId, err.message); }
  }
  $("runBattingML").addEventListener("click", () => prediction("/api/predict/batting/", "battingPlayer", "battingJson", "battingMLResult"));
  $("runBowlingML").addEventListener("click", () => prediction("/api/predict/bowling/", "bowlingPlayer", "bowlingJson", "bowlingMLResult"));

  $("loadMetrics").addEventListener("click", async () => {
    try { show("metricsResult", await request("/api/eda/model-metrics/")); }
    catch (err) { show("metricsResult", "Saved metrics could not be loaded: " + err.message); }
  });
  $("loadEDA").addEventListener("click", async () => {
    try { show("edaResult", await request($("edaEndpoint").value)); }
    catch (err) { show("edaResult", err.message); }
  });
  $("runBatch").addEventListener("click", async () => {
    try {
      const payload = safeParse($("batchJson").value, "Batch payload");
      const data = await request($("batchEndpoint").value, {method:"POST", body:JSON.stringify(payload)});
      show("batchResult", data);
      const exportButton = document.getElementById("downloadBatch");
      if (exportButton) exportButton.onclick = () => {
        const blob = new Blob([JSON.stringify(data, null, 2)], {type:"application/json"});
        const url = URL.createObjectURL(blob);
        const a = document.createElement("a"); a.href=url; a.download="ipl_batch_results.json"; a.click();
        URL.revokeObjectURL(url);
      };
      if (exportButton) exportButton.hidden = false;
    } catch (err) { show("batchResult", err.message); }
  });
  $("checkStatus").addEventListener("click", async () => {
    const paths = ["/api/health/", "/api/ingestion/status/"];
    const results = {};
    for (const path of paths) {
      try { results[path] = {available:true, response:await request(path)}; }
      catch (err) { results[path] = {available:false, detail:err.message}; }
    }
    show("statusResult", results);
  });
})();'''
JS_PATH.write_text(js, encoding="utf-8")
print("Wrote:", JS_PATH.relative_to(ROOT))


Wrote: django_api\ipl_api\static\ipl_api\js\dashboard_day25.js


## 4. Add responsive styling

In [5]:
css = r'''body { background: #f3f5f9; color: #172033; }
main { max-width: 1800px; }
.feature-section { display: none; }
.feature-section.active { display: block; }
.metric-card { border: 0; box-shadow: 0 2px 10px rgba(20, 35, 65, .07); min-height: 120px; }
.metric-value { font-size: clamp(1.7rem, 3vw, 2.4rem); font-weight: 700; overflow-wrap: anywhere; }
.card { border: 1px solid #e1e6ef; box-shadow: 0 2px 8px rgba(20, 35, 65, .04); }
.card-header { background: #fff; font-weight: 600; }
.code-input { min-height: 160px; font-family: Consolas, "Courier New", monospace; font-size: .9rem; }
.result-box { min-height: 110px; max-height: 430px; overflow: auto; white-space: pre-wrap; overflow-wrap: anywhere; background: #101827; color: #e8eefb; padding: 1rem; border-radius: .5rem; font-size: .88rem; }
.nav-pills .nav-link { background: #e4e9f2; color: #27344b; }
.nav-pills .nav-link.active { background: #0d6efd; color: #fff; }
@media (max-width: 767px) { main { padding-left: 12px !important; padding-right: 12px !important; } .code-input { min-height: 130px; } }
'''
CSS_PATH.write_text(css, encoding="utf-8")
print("Wrote:", CSS_PATH.relative_to(ROOT))


Wrote: django_api\ipl_api\static\ipl_api\css\dashboard_day25.css


In [6]:
# Validate URL registration and Django configuration.
check = subprocess.run(
    [sys.executable, "manage.py", "check"],
    cwd=DJANGO_DIR, capture_output=True, text=True
)
print(check.stdout)
if check.stderr:
    print("STDERR:\n", check.stderr)
print("Django check return code:", check.returncode)
if check.returncode != 0:
    raise RuntimeError("Django system check failed.")

resolve_code = r"""
from django.urls import resolve
paths = [
    "/api/dashboard/", "/api/predictions/", "/api/players/",
    "/api/players/search-options/", "/api/eda/summary/",
    "/api/eda/batting/", "/api/eda/bowling/", "/api/eda/season/",
    "/api/eda/venue/", "/api/eda/opposition/", "/api/eda/recent-form/",
    "/api/eda/model-metrics/", "/api/predict/", "/api/predict/batting/",
    "/api/predict/bowling/", "/api/predict/batch/", "/api/predict/player/",
]
for path in paths:
    match = resolve(path)
    print(path, "->", match.url_name, match.func.__module__)
"""
resolved = subprocess.run(
    [sys.executable, "manage.py", "shell", "-c", resolve_code],
    cwd=DJANGO_DIR, capture_output=True, text=True
)
print(resolved.stdout)
if resolved.stderr:
    print("URL resolution STDERR:\n", resolved.stderr)
print("URL resolution return code:", resolved.returncode)
if resolved.returncode != 0:
    raise RuntimeError("URL resolution failed.")


System check identified no issues (0 silenced).

Django check return code: 0
12 objects imported automatically (use -v 2 for details).

/api/dashboard/ -> ipl-dashboard ipl_api.dashboard_views_day26
/api/predictions/ -> ipl-predictions ipl_api.prediction_center_views_day28
/api/players/ -> player-list ipl_api.views
/api/players/search-options/ -> player-name-options-day28 ipl_api.player_name_prediction_views_day28
/api/eda/summary/ -> eda-summary ipl_api.eda_summary_day26
/api/eda/batting/ -> eda-batting ipl_api.eda_views
/api/eda/bowling/ -> eda-bowling ipl_api.eda_views
/api/eda/season/ -> eda-season ipl_api.eda_views
/api/eda/venue/ -> eda-venue ipl_api.eda_views
/api/eda/opposition/ -> eda-opposition ipl_api.eda_views
/api/eda/recent-form/ -> eda-recent-form ipl_api.eda_views
/api/eda/model-metrics/ -> model-metrics ipl_api.player_name_prediction_views_day28
/api/predict/ -> predict-unified ipl_api.prediction_views
/api/predict/batting/ -> predict-batting ipl_api.prediction_views
/

## 6. Validate generated files, Django settings, and route resolution

In [7]:
# Syntax-check the JavaScript using Node if installed; otherwise record the skip.
node = shutil.which("node")
if node:
    js_check = subprocess.run([node, "--check", str(JS_PATH)], capture_output=True, text=True)
    print(js_check.stdout)
    if js_check.stderr:
        print(js_check.stderr)
    print("Node syntax-check return code:", js_check.returncode)
    if js_check.returncode != 0:
        raise RuntimeError("JavaScript syntax check failed.")
else:
    print("Node.js not found; skipped JavaScript syntax check.")

check = subprocess.run(
    [sys.executable, "manage.py", "check"],
    cwd=DJANGO_DIR,
    capture_output=True,
    text=True,
)
print(check.stdout)
if check.stderr:
    print("STDERR:\n", check.stderr)
print("Django check return code:", check.returncode)
if check.returncode != 0:
    raise RuntimeError("Django system check failed.")


Node.js not found; skipped JavaScript syntax check.
System check identified no issues (0 silenced).

Django check return code: 0


## 7. Acceptance checklist

## Auto-generate documentation for every file created or modified

This cell creates:
- `docs/day30/day30_dashboard_features_report.md` — overall report
- `docs/day30/generated_files_manifest.csv` — one row per generated/modified file
- `docs/day30/<filename>.md` — an individual documentation file for each generated dashboard asset

It uses the actual paths and current file contents from this run. It does not claim endpoints work just because files exist.

In [10]:
from pathlib import Path
import datetime
import csv
import hashlib

DOC_DIR = ROOT / "docs" 
DOC_DIR.mkdir(parents=True, exist_ok=True)
generated_paths = [
    CONFIG_URLS,
    URLS_PATH,
    TEMPLATE_PATH,
    JS_PATH,
    CSS_PATH,
]
# Include the existing route file and dashboard view as inspected dependencies, but do not modify them here.
dependency_paths = [DASHBOARD_VIEW_PATH, PREDICTION_VIEW_PATH]
timestamp = datetime.datetime.now().isoformat(timespec="seconds")

manifest_path = DOC_DIR / "generated_files_manifest.csv"
rows = []
for file_path in generated_paths + dependency_paths:
    exists = file_path.exists()
    content = file_path.read_bytes() if exists else b""
    rows.append({
        "path": str(file_path.relative_to(ROOT)),
        "role": "generated_or_modified" if file_path in generated_paths else "inspected_dependency",
        "exists": exists,
        "bytes": len(content),
        "sha256": hashlib.sha256(content).hexdigest() if exists else "",
    })

with manifest_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["path", "role", "exists", "bytes", "sha256"])
    writer.writeheader()
    writer.writerows(rows)

individual_docs = []
for file_path in generated_paths:
    exists = file_path.exists()
    content = file_path.read_text(encoding="utf-8") if exists else ""
    doc_path = DOC_DIR / (file_path.name + ".md")
    summary = {
        TEMPLATE_PATH: "Django HTML template for the eight dashboard sections and their controls.",
        JS_PATH: "Browser JavaScript for tab navigation, API requests, result rendering, and batch JSON export.",
        CSS_PATH: "Responsive styling for dashboard sections, cards, inputs, and result panels.",
    }.get(file_path, "Generated project asset.")
    doc = f"""# File documentation: `{file_path.relative_to(ROOT)}`

- Generated: {timestamp}
- Exists at documentation time: {exists}
- Bytes: {len(content.encode("utf-8")) if exists else 0}
- Purpose: {summary}

## Key elements
"""
    if exists:
        if file_path.suffix == ".html":
            ids = re.findall(r'id="([^"]+)"', content)
            doc += "- HTML element IDs: " + (", ".join(f"`{x}`" for x in ids) if ids else "none detected") + "\n"
            doc += "- Template includes Django static-file loading and Bootstrap assets.\n"
        elif file_path.suffix == ".js":
            funcs = re.findall(r'async function ([A-Za-z0-9_]+)|function ([A-Za-z0-9_]+)', content)
            names = sorted({a or b for a, b in funcs})
            doc += "- Functions: " + (", ".join(f"`{x}`" for x in names) if names else "inline event handlers / arrow functions") + "\n"
            doc += "- Uses fetch for registered API calls and displays HTTP errors in the UI.\n"
        elif file_path.suffix == ".css":
            selectors = re.findall(r'(?m)^([^@\\s][^{]+)\\s*\\{', content)
            doc += "- CSS selectors: " + (", ".join(f"`{x.strip()}`" for x in selectors[:40]) if selectors else "none detected") + "\n"
        doc += "\n## File content\n```" + ("html" if file_path.suffix == ".html" else "javascript" if file_path.suffix == ".js" else "css") + "\n" + content + "\n```\n"
    else:
        doc += "\nFile was not present when this report was generated.\n"
    doc_path.write_text(doc, encoding="utf-8")
    individual_docs.append(doc_path)

report_path = DOC_DIR / "day30_dashboard_features_report.md"
report = f"""# Day 30 — Dashboard Features and Integration Report

- Generated: {timestamp}
- Repository: `{ROOT}`
- Notebook: `30_ipl_dashboard_requested_features.ipynb`

## Requested dashboard sections
1. Dashboard Overview — live summary endpoint values.
2. Player Search — player-name lookup and historical response.
3. Batting Predictions — separate ML and DL panels.
4. Bowling Predictions — separate ML and DL panels.
5. Model Comparison — saved metrics endpoint.
6. EDA & Analytics — summary, batting, bowling, season, venue, opposition, recent form.
7. Batch Analysis — JSON request and downloadable JSON response.
8. Data Ingestion & Status — checks endpoint availability without claiming ingestion ran.

## Generated/modified assets
""" + "\n".join(f"- `{x['path']}` — role: {x['role']}; exists: {x['exists']}; bytes: {x['bytes']}" for x in rows) + """

## Validation interpretation
File existence and `manage.py check` do not prove that browser workflows, database queries, model inference, or downloads work. Use the actual notebook output and browser acceptance checklist to record validation.

## Known dependencies and limitations
- Prediction requests must follow the real backend schema and saved model manifest.
- A dedicated DL endpoint is required before enabling LSTM/GRU inference.
- Ingestion run/status endpoints must exist before ingestion controls can perform actions.
- The metrics endpoint must return real saved evaluation values before model comparison can be considered complete.

## Individual file documentation
""" + "\n".join(f"- `{x.relative_to(ROOT)}`" for x in individual_docs) + f"""

## Manifest
- `{manifest_path.relative_to(ROOT)}`
"""
report_path.write_text(report, encoding="utf-8")
print("Generated overall report:", report_path)
print("Generated file manifest:", manifest_path)
print("Generated per-file docs:")
for doc in individual_docs:
    print(" -", doc)


Generated overall report: D:\Git\Personal\ipl-player-performance\docs\day30_dashboard_features_report.md
Generated file manifest: D:\Git\Personal\ipl-player-performance\docs\generated_files_manifest.csv
Generated per-file docs:
 - D:\Git\Personal\ipl-player-performance\docs\urls.py.md
 - D:\Git\Personal\ipl-player-performance\docs\urls.py.md
 - D:\Git\Personal\ipl-player-performance\docs\dashboard_day25.html.md
 - D:\Git\Personal\ipl-player-performance\docs\dashboard_day25.js.md
 - D:\Git\Personal\ipl-player-performance\docs\dashboard_day25.css.md
